# Zotero Group Highlights Exporter

Dieses Notebook extrahiert alle Highlights aus PDFs einer Zotero-Gruppe und exportiert sie in eine Excel-Datei.

## Installation der benötigten Pakete
Führe diese Zelle einmalig aus:

In [ ]:
!pip install requests pandas openpyxl ipywidgets tqdm

## Import der Bibliotheken

In [14]:
import requests
import pandas as pd
import os
from typing import List, Dict, Optional
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

## Konfiguration - Eingabefelder

In [15]:
# Eingabe-Widgets erstellen
group_id_widget = widgets.Text(
    value='',
    placeholder='z.B. 1234567',
    description='Group ID:',
    style={'description_width': 'initial'}
)

api_key_widget = widgets.Password(
    value='',
    placeholder='Dein Zotero API Key',
    description='API Key:',
    style={'description_width': 'initial'}
)

output_path_widget = widgets.Text(
    value='zotero_highlights.xlsx',
    placeholder='Pfad zur Excel-Datei',
    description='Ausgabepfad:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='70%')
)

filter_tag_widget = widgets.Text(
    value='',
    placeholder='Optional: Nur Items mit diesem Tag (leer = alle)',
    description='Filter Tag:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='70%')
)

# Anzeige
display(widgets.VBox([
    widgets.HTML('<h3>Zotero API Konfiguration</h3>'),
    group_id_widget,
    api_key_widget,
    output_path_widget,
    filter_tag_widget,
    widgets.HTML('<p style="color: gray;"><i>Tipp: Du findest deine Group ID in der URL deiner Zotero-Gruppe</i></p>')
]))

## Haupt-Klasse: Zotero Highlights Exporter

In [16]:
class ZoteroHighlightsExporter:
    """Exportiert Highlights aus einer Zotero-Gruppe in eine Excel-Datei"""
    
    def __init__(self, group_id: str, api_key: str):
        self.group_id = group_id
        self.api_key = api_key
        self.base_url = f"https://api.zotero.org/groups/{group_id}"
        self.headers = {
            'Zotero-API-Key': api_key,
            'User-Agent': 'ZoteroHighlightsExporter/1.0'
        }
        self.session = requests.Session()
        self.session.headers.update(self.headers)
    
    def _make_request(self, endpoint: str, params: dict = None) -> tuple:
        """Führt API-Request mit Fehlerbehandlung aus"""
        url = f"{self.base_url}{endpoint}"
        try:
            response = self.session.get(url, params=params, timeout=30)
            response.raise_for_status()
            return response.json(), response.headers
        except requests.exceptions.RequestException as e:
            print(f"❌ Fehler bei API-Request: {e}")
            return [], {}
    
    def _get_all_items(self, endpoint: str, params: dict = None) -> List[Dict]:
        """Lädt alle Items mit Paginierung"""
        all_items = []
        start = 0
        limit = 100
        
        if params is None:
            params = {}
        
        while True:
            params.update({'start': start, 'limit': limit})
            items, headers = self._make_request(endpoint, params)
            
            if not items:
                break
            
            all_items.extend(items)
            
            # Prüfe ob es mehr Items gibt
            total = headers.get('Total-Results')
            if total and int(total) <= start + limit:
                break
            
            start += limit
            time.sleep(0.1)  # Rate limiting
        
        return all_items
    
    def get_top_level_items(self, filter_tag: Optional[str] = None) -> List[Dict]:
        """Lädt alle Top-Level Items (keine Attachments/Notes)"""
        print("📥 Lade Top-Level Items...")
        params = {}
        if filter_tag:
            params['tag'] = filter_tag
        
        items = self._get_all_items('/items/top', params)
        print(f"✅ {len(items)} Items gefunden")
        return items
    
    def get_pdf_attachments(self, item_key: str) -> List[Dict]:
        """Lädt PDF-Attachments für ein Item"""
        endpoint = f'/items/{item_key}/children'
        params = {'itemType': 'attachment'}
        attachments = self._get_all_items(endpoint, params)
        
        # Filtere nur PDFs
        pdfs = [
            att for att in attachments 
            if att.get('data', {}).get('contentType') == 'application/pdf'
        ]
        return pdfs
    
    def get_annotations(self, item_key: str) -> List[Dict]:
        """Lädt Annotationen für ein Attachment"""
        endpoint = f'/items/{item_key}/children'
        params = {'itemType': 'annotation'}
        return self._get_all_items(endpoint, params)
    
    def extract_authors(self, item_data: Dict) -> str:
        """Extrahiert Autoren im Format 'Nachname, Vorname'"""
        creators = item_data.get('creators', [])
        authors = []
        
        for creator in creators:
            if creator.get('creatorType') == 'author':
                last = creator.get('lastName', '')
                first = creator.get('firstName', '')
                if last:
                    authors.append(f"{last}, {first}" if first else last)
        
        return '; '.join(authors) if authors else 'N/A'
    
    def extract_tags(self, annotation_data: Dict) -> str:
        """Extrahiert Tags einer Annotation"""
        tags = annotation_data.get('tags', [])
        return ', '.join([tag.get('tag', '') for tag in tags])
    
    def export_highlights(self, output_path: str, filter_tag: Optional[str] = None):
        """Hauptfunktion: Exportiert alle Highlights in Excel"""
        print("\n" + "="*60)
        print("🚀 Starte Zotero Highlights Export")
        print("="*60 + "\n")
        
        # Validierung
        output_dir = os.path.dirname(output_path)
        if output_dir and not os.path.exists(output_dir):
            print(f"❌ Verzeichnis existiert nicht: {output_dir}")
            return
        
        # Lade Items
        top_items = self.get_top_level_items(filter_tag)
        if not top_items:
            print("❌ Keine Items gefunden!")
            return
        
        # Sammle Highlights
        highlights_data = []
        seen_highlights = set()  # Duplikate vermeiden
        
        print(f"\n🔍 Durchsuche {len(top_items)} Items nach Highlights...\n")
        
        for item in tqdm(top_items, desc="Items"):
            item_data = item.get('data', {})
            item_key = item.get('key')
            
            # Metadaten extrahieren
            paper_title = item_data.get('title', 'N/A')
            authors = self.extract_authors(item_data)
            
            # PDF-Attachments laden
            pdfs = self.get_pdf_attachments(item_key)
            
            for pdf in pdfs:
                pdf_key = pdf.get('key')
                
                # Annotationen laden
                annotations = self.get_annotations(pdf_key)
                
                for ann in annotations:
                    ann_data = ann.get('data', {})
                    
                    # Nur Highlights (keine Comments/Images)
                    if ann_data.get('annotationType') != 'highlight':
                        continue
                    
                    highlight_text = ann_data.get('annotationText', '').strip()
                    if not highlight_text:
                        continue
                    
                    # WICHTIG: Comment extrahieren
                    comment = ann_data.get('annotationComment', '').strip()
                    page = ann_data.get('annotationPageLabel', '')
                    tags = self.extract_tags(ann_data)
                    
                    # Duplikat-Check
                    signature = (paper_title, highlight_text, page)
                    if signature in seen_highlights:
                        continue
                    seen_highlights.add(signature)
                    
                    # Daten sammeln - MIT COMMENT!
                    highlights_data.append({
                        'Paper Title': paper_title,
                        'Authors': authors,
                        'Highlighted Text': highlight_text,
                        'Comment': comment,
                        'Page': page,
                        'Tags': tags
                    })
        
        # Export
        if not highlights_data:
            print("\n⚠️  Keine Highlights gefunden!")
            return
        
        print(f"\n💾 Exportiere {len(highlights_data)} Highlights...")
        
        df = pd.DataFrame(highlights_data)
        
        try:
            df.to_excel(output_path, index=False, engine='openpyxl')
            print(f"\n✅ Export erfolgreich!")
            print(f"📁 Datei gespeichert: {os.path.abspath(output_path)}")
            print(f"📊 Anzahl Highlights: {len(highlights_data)}")
            print(f"📄 Anzahl Papers: {df['Paper Title'].nunique()}")
            print(f"💬 Highlights mit Kommentaren: {df['Comment'].notna().sum()}")
            
            # Vorschau
            print("\n👀 Vorschau der ersten 3 Highlights:")
            print("="*60)
            display(df.head(3))
            
        except Exception as e:
            print(f"\n❌ Fehler beim Speichern: {e}")

## Export durchführen

**Führe diese Zelle aus, nachdem du oben die Konfiguration eingegeben hast:**

In [17]:
# Werte aus Widgets holen
GROUP_ID = group_id_widget.value.strip()
API_KEY = api_key_widget.value.strip()
OUTPUT_PATH = output_path_widget.value.strip()
FILTER_TAG = filter_tag_widget.value.strip() or None

# Validierung
if not GROUP_ID or not API_KEY:
    print("❌ Bitte fülle Group ID und API Key aus!")
else:
    # Exporter initialisieren und ausführen
    exporter = ZoteroHighlightsExporter(GROUP_ID, API_KEY)
    exporter.export_highlights(OUTPUT_PATH, FILTER_TAG)


🚀 Starte Zotero Highlights Export

📥 Lade Top-Level Items...
✅ 35 Items gefunden

🔍 Durchsuche 35 Items nach Highlights...



Items:   0%|          | 0/35 [00:00<?, ?it/s]


💾 Exportiere 46 Highlights...

✅ Export erfolgreich!
📁 Datei gespeichert: /Users/korbinian/jupyter/ZoteroPlugin/zotero_highlights_2.xlsx
📊 Anzahl Highlights: 46
📄 Anzahl Papers: 7
💬 Highlights mit Kommentaren: 46

👀 Vorschau der ersten 3 Highlights:


,Paper Title,Authors,Highlighted Text,Comment,Page,Tags
0,A review of methods and applications in struct...,"Zhang, Bangcheng; Ren, Yuheng; He, Siming; Gao...",Data preprocessing in bridge SHM,,17,"data pereparation, data quality"
1,A review of methods and applications in struct...,"Zhang, Bangcheng; Ren, Yuheng; He, Siming; Gao...",Quintana et al. analyzed the technical and eco...,,16,costs
2,A review of methods and applications in struct...,"Zhang, Bangcheng; Ren, Yuheng; He, Siming; Gao...",various emerging technologies tend to generate...,,16,costs


## Zusätzliche Analyse (Optional)

Lade die exportierte Excel-Datei und analysiere die Highlights:

In [ ]:
# Excel-Datei laden
try:
    df = pd.read_excel(OUTPUT_PATH)
    
    print("📊 Statistiken:")
    print(f"   Gesamte Highlights: {len(df)}")
    print(f"   Anzahl Papers: {df['Paper Title'].nunique()}")
    print(f"   Anzahl eindeutige Autoren: {df['Authors'].nunique()}")
    print(f"   Highlights MIT Kommentar: {df['Comment'].notna().sum()} ({df['Comment'].notna().sum()/len(df)*100:.1f}%)")
    print(f"\n🏆 Top 5 Papers mit den meisten Highlights:")
    print(df['Paper Title'].value_counts().head())
    
    print(f"\n🔖 Häufigste Tags:")
    all_tags = []
    for tags in df['Tags'].dropna():
        if tags:
            all_tags.extend([t.strip() for t in tags.split(',')])
    if all_tags:
        tag_series = pd.Series(all_tags)
        print(tag_series.value_counts().head())
    else:
        print("   Keine Tags gefunden")
    
    print(f"\n💬 Beispiel-Kommentare:")
    comments_df = df[df['Comment'].notna() & (df['Comment'] != '')]
    if len(comments_df) > 0:
        for idx, row in comments_df.head(3).iterrows():
            print(f"\n   Paper: {row['Paper Title'][:50]}...")
            print(f"   Highlight: {row['Highlighted Text'][:80]}...")
            print(f"   Comment: {row['Comment']}")
    else:
        print("   Keine Kommentare gefunden")
        
except FileNotFoundError:
    print("❌ Excel-Datei nicht gefunden. Führe zuerst den Export aus!")
except Exception as e:
    print(f"❌ Fehler beim Laden: {e}")

## Troubleshooting

### Häufige Probleme:

1. **"Keine Items gefunden"**
   - Prüfe die Group ID (findest du in der URL: `zotero.org/groups/XXXXXX`)
   - Prüfe die API-Key-Berechtigungen (muss Lesezugriff auf die Gruppe haben)

2. **"Keine Highlights gefunden"**
   - Stelle sicher, dass PDFs in Zotero mit dem nativen Reader geöffnet wurden
   - Highlights müssen vom Typ "highlight" sein (keine Notizen/Kommentare)

3. **"Permission denied" beim Speichern**
   - Prüfe Schreibrechte im Zielverzeichnis
   - Schließe die Excel-Datei, falls sie bereits geöffnet ist

### API-Key erstellen:
1. Gehe zu https://www.zotero.org/settings/keys
2. Erstelle einen neuen Key mit **"Allow library access"** für die Gruppe
3. Kopiere den Key (er wird nur einmal angezeigt!)

### Excel-Spalten:
Die exportierte Datei enthält folgende Spalten:
- **Paper Title**: Titel des Artikels
- **Authors**: Autoren (Nachname, Vorname)
- **Highlighted Text**: Der markierte Text
- **Comment**: Dein Kommentar zum Highlight (falls vorhanden)
- **Page**: Seitenzahl
- **Tags**: Alle Tags des Highlights